In [1]:
import torch
import torchvision
from pathlib import Path

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM (GB):",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    )

print("\n=== DATASET FILES ===")
for path in Path("/kaggle/input").rglob("*"):
    if path.is_file():
        print(path)
        if path.suffix.lower() == ".jpg":
            break

PyTorch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM (GB): 14.56

=== DATASET FILES ===
/kaggle/input/datasets/chienvoo/the-imposter-data/the_imposter/baseline_TACVU1.ipynb
/kaggle/input/datasets/chienvoo/the-imposter-data/the_imposter/data/public_test/pairs.csv
/kaggle/input/datasets/chienvoo/the-imposter-data/the_imposter/data/train/pairs.csv
/kaggle/input/datasets/chienvoo/the-imposter-data/the_imposter/data/train/images/00001.jpg


In [4]:
from pathlib import Path
import pandas as pd

# Tìm chính xác pairs.csv thuộc thư mục train
csv_files = [
    p for p in Path("/kaggle/input").rglob("pairs.csv")
    if p.parent.name == "train"
]

assert len(csv_files) == 1, (
    f"Expected 1 train/pairs.csv, found {len(csv_files)}"
)

PAIRS_CSV = csv_files[0]
TRAIN_DIR = PAIRS_CSV.parent

print("TRAIN_DIR:", TRAIN_DIR)
print("PAIRS_CSV:", PAIRS_CSV)

# Đọc dữ liệu huấn luyện
pairs_df = pd.read_csv(PAIRS_CSV, dtype={"pair_id": str})

print("\nShape:", pairs_df.shape)
print("Columns:", pairs_df.columns.tolist())

display(pairs_df.head())

# Kiểm tra ảnh
first_image = TRAIN_DIR / pairs_df.iloc[0]["image_0"]

print("\nFirst image:", first_image)
print("Image exists:", first_image.exists())

TRAIN_DIR: /kaggle/input/datasets/chienvoo/the-imposter-data/the_imposter/data/train
PAIRS_CSV: /kaggle/input/datasets/chienvoo/the-imposter-data/the_imposter/data/train/pairs.csv

Shape: (1000, 4)
Columns: ['pair_id', 'image_0', 'image_1', 'fake_position']


,pair_id,image_0,image_1,fake_position
0,00001,images/00295.jpg,images/01131.jpg,1
1,00002,images/00684.jpg,images/00888.jpg,0
2,00003,images/00016.jpg,images/00059.jpg,0
3,00004,images/01000.jpg,images/00449.jpg,1
4,00005,images/01117.jpg,images/01089.jpg,1



First image: /kaggle/input/datasets/chienvoo/the-imposter-data/the_imposter/data/train/images/00295.jpg
Image exists: True


In [7]:
# Tìm file development_split.csv
split_files = list(
    Path("/kaggle/input").rglob("development_split.csv")
)

print("Found split files:", split_files)

assert len(split_files) == 1, (
    f"Expected 1 development_split.csv, found {len(split_files)}"
)

split_df = pd.read_csv(
    split_files[0],
    dtype={"pair_id": str}
)

print("\nSplit shape:", split_df.shape)
print("\nSplit counts:")
print(split_df["split"].value_counts())

print("\nFold counts:")
print(split_df.groupby(["split", "fold"]).size())

display(split_df.head())

Found split files: [PosixPath('/kaggle/input/datasets/chienvoo/development-split/development_split.csv')]

Split shape: (1000, 3)

Split counts:
split
development    800
holdout        200
Name: count, dtype: int64

Fold counts:
split        fold
development   0      267
              1      267
              2      266
holdout      -1      200
dtype: int64


,pair_id,split,fold
0,00720,development,1
1,00610,development,2
2,00501,development,2
3,00796,development,0
4,00576,development,0


In [8]:
# Ghép thông tin ảnh với development split
merged_df = pairs_df.merge(
    split_df,
    on="pair_id",
    how="inner",
    validate="one_to_one"
)

assert len(merged_df) == 1000

dev_df = (
    merged_df[merged_df["split"] == "development"]
    .sort_values("pair_id")
    .reset_index(drop=True)
)

print("Merged:", merged_df.shape)
print("Development:", dev_df.shape)
print("Holdout:", (merged_df["split"] == "holdout").sum())

print("\nLabels:")
print(dev_df["fake_position"].value_counts())

display(dev_df.head())

Merged: (1000, 6)
Development: (800, 6)
Holdout: 200

Labels:
fake_position
1    400
0    400
Name: count, dtype: int64


,pair_id,image_0,image_1,fake_position,split,fold
0,00001,images/00295.jpg,images/01131.jpg,1,development,1
1,00002,images/00684.jpg,images/00888.jpg,0,development,1
2,00004,images/01000.jpg,images/00449.jpg,1,development,1
3,00005,images/01117.jpg,images/01089.jpg,1,development,2
4,00006,images/01265.jpg,images/01659.jpg,0,development,0


In [9]:
import torch
from torchvision.models import densenet121, DenseNet121_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Trọng số pretrained trên ImageNet
weights = DenseNet121_Weights.IMAGENET1K_V1

model = densenet121(weights=weights)

# Bỏ tầng phân loại 1000 lớp để lấy embedding 1024 chiều
model.classifier = torch.nn.Identity()

# Đóng băng toàn bộ tham số
for param in model.parameters():
    param.requires_grad = False

model = model.to(device)
model.eval()

print("Device:", device)
print("Model:", model.__class__.__name__)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Feature dimension:", model.classifier)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 179MB/s]


Device: cuda
Model: DenseNet
Trainable parameters: 0
Feature dimension: Identity()


In [10]:
import numpy as np
from PIL import Image

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Tiền xử lý theo ImageNet
transform = weights.transforms()

class FaceImageDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]

        with Image.open(image_path) as img:
            image = img.convert("RGB")
            image = self.transform(image)

        return image, str(image_path)


# Lấy danh sách ảnh từ 800 cặp development
image_paths = sorted(
    set(
        [str(TRAIN_DIR / p) for p in dev_df["image_0"]]
        +
        [str(TRAIN_DIR / p) for p in dev_df["image_1"]]
    )
)

dataset = FaceImageDataset(
    image_paths=image_paths,
    transform=transform
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Number of development pairs:", len(dev_df))
print("Number of unique images:", len(dataset))

# Kiểm tra một batch
images, paths = next(iter(loader))

print("Batch shape:", images.shape)
print("First image:", paths[0])

Number of development pairs: 800
Number of unique images: 1600
Batch shape: torch.Size([32, 3, 224, 224])
First image: /kaggle/input/datasets/chienvoo/the-imposter-data/the_imposter/data/train/images/00001.jpg


In [11]:
from tqdm.auto import tqdm
import numpy as np
import torch

model.eval()

feature_dict = {}

with torch.inference_mode():
    for batch_images, batch_paths in tqdm(
        loader,
        desc="Extracting DenseNet121 features"
    ):
        batch_images = batch_images.to(
            device,
            non_blocking=True
        )

        embeddings = model(batch_images)

        embeddings = embeddings.cpu().numpy()

        for path, embedding in zip(batch_paths, embeddings):
            feature_dict[path] = embedding

print("Extracted images:", len(feature_dict))

first_embedding = next(iter(feature_dict.values()))

print("Embedding shape:", first_embedding.shape)
print("Contains NaN:", np.isnan(first_embedding).any())
print("Contains Inf:", np.isinf(first_embedding).any())

Extracting DenseNet121 features:   0%|          | 0/50 [00:00<?, ?it/s]

Extracted images: 1600
Embedding shape: (1024,)
Contains NaN: False
Contains Inf: False


In [13]:
X_cnn = np.stack([
    feature_dict[str(TRAIN_DIR / row.image_1)]
    - feature_dict[str(TRAIN_DIR / row.image_0)]
    for row in dev_df.itertuples(index=False)
]).astype(np.float32)

y_cnn = dev_df["fake_position"].to_numpy(dtype=np.int64)
folds_cnn = dev_df["fold"].to_numpy(dtype=np.int64)

print("X_cnn:", X_cnn.shape)
print("y_cnn:", y_cnn.shape)
print("folds_cnn:", folds_cnn.shape)

print("NaN:", np.isnan(X_cnn).any())
print("Inf:", np.isinf(X_cnn).any())

X_cnn: (800, 1024)
y_cnn: (800,)
folds_cnn: (800,)
NaN: False
Inf: False


In [14]:
from pathlib import Path

CACHE_DIR = Path("/kaggle/working/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

cache_path = CACHE_DIR / "development_densenet121_frozen.npz"

np.savez_compressed(
    cache_path,
    X=X_cnn,
    y=y_cnn,
    folds=folds_cnn,
    pair_ids=dev_df["pair_id"].to_numpy()
)

print("Saved:", cache_path)
print("File size (MB):", round(cache_path.stat().st_size / 1024**2, 2))

Saved: /kaggle/working/cache/development_densenet121_frozen.npz
File size (MB): 2.99


In [15]:
import numpy as np

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)

oof_prob_cnn = np.zeros(len(y_cnn), dtype=np.float64)
oof_pred_cnn = np.zeros(len(y_cnn), dtype=np.int64)

for fold in sorted(np.unique(folds_cnn)):
    train_idx = folds_cnn != fold
    valid_idx = folds_cnn == fold

    clf = make_pipeline(
        StandardScaler(with_mean=False),
        LogisticRegression(
            C=0.1,
            fit_intercept=False,
            max_iter=5000,
            random_state=42
        )
    )

    clf.fit(X_cnn[train_idx], y_cnn[train_idx])

    prob = clf.predict_proba(X_cnn[valid_idx])[:, 1]
    pred = (prob >= 0.5).astype(np.int64)

    oof_prob_cnn[valid_idx] = prob
    oof_pred_cnn[valid_idx] = pred

    fold_f1 = f1_score(
        y_cnn[valid_idx],
        pred,
        average="macro"
    )

    print(
        f"Fold {fold}: "
        f"Macro-F1 = {fold_f1:.4f}, "
        f"Samples = {valid_idx.sum()}"
    )

print("\n=== OVERALL OOF ===")

macro_f1_cnn = f1_score(
    y_cnn,
    oof_pred_cnn,
    average="macro"
)

accuracy_cnn = accuracy_score(y_cnn, oof_pred_cnn)
errors_cnn = int(np.sum(y_cnn != oof_pred_cnn))

print(f"Macro-F1: {macro_f1_cnn:.4f}")
print(f"Accuracy: {accuracy_cnn:.4f}")
print(f"Errors: {errors_cnn}/{len(y_cnn)}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_cnn, oof_pred_cnn))

print("\nClassification Report:")
print(classification_report(y_cnn, oof_pred_cnn))

Fold 0: Macro-F1 = 0.8502, Samples = 267
Fold 1: Macro-F1 = 0.8277, Samples = 267
Fold 2: Macro-F1 = 0.8720, Samples = 266

=== OVERALL OOF ===
Macro-F1: 0.8500
Accuracy: 0.8500
Errors: 120/800

Confusion Matrix:
[[344  56]
 [ 64 336]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.86      0.85       400
           1       0.86      0.84      0.85       400

    accuracy                           0.85       800
   macro avg       0.85      0.85      0.85       800
weighted avg       0.85      0.85      0.85       800



In [16]:
import pandas as pd

comparison_df = pd.DataFrame({
    "Method": [
        "Handcrafted 32 features + LR",
        "Frozen DenseNet121 features + LR"
    ],
    "Macro-F1": [
        0.7862,
        macro_f1_cnn
    ],
    "Errors": [
        171,
        errors_cnn
    ]
})

display(comparison_df)

print(
    "Macro-F1 difference:",
    f"{(macro_f1_cnn - 0.7862) * 100:+.2f} percentage points"
)

,Method,Macro-F1,Errors
0,Handcrafted 32 features + LR,0.786200,171
1,Frozen DenseNet121 features + LR,0.849985,120


Macro-F1 difference: +6.38 percentage points


In [18]:
RESULTS_DIR = Path("/kaggle/working/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

oof_df = pd.DataFrame({
    "pair_id": dev_df["pair_id"],
    "fold": folds_cnn,
    "fake_position": y_cnn,
    "prob_fake_1": oof_prob_cnn,
    "prediction": oof_pred_cnn,
    "correct": oof_pred_cnn == y_cnn
})

result_path = RESULTS_DIR / "frozen_densenet121_oof.csv"

oof_df.to_csv(result_path, index=False)

print("Saved:", result_path)
print("Rows:", len(oof_df))

Saved: /kaggle/working/results/frozen_densenet121_oof.csv
Rows: 800
